# Trend Strength Phase 1
固定28戦略・16,298 tradesの純粋診断。Entry/Risk/live/Volatility Phase 5変更なし。
計画SHA: `50e7d70cf1eb1fe892c144a0721dbf490e07b656`（実装前remote確認済み）。
Primary: Wilder D1 ADX14。Robustness: ER20（分母0は0）。直前252完了日midrank、LOW p<1/3、NORMAL 1/3<=p<2/3、HIGH p>=2/3。
正式支持: pooled HIGH-LOW 95%週cluster CIが0除外、等重み同符号、十分標本戦略の厳密過半数同符号。両方式同方向支持でBOTH_SUPPORTED。
各cell20未満LOW_SAMPLE、等重みはLOW/HIGH双方20以上の同一戦略集合。NORMALは記述。
結果計算前の実装版。実行後の結果は次の表示セルに出力されます。


In [ ]:
from pathlib import Path
RESEARCH_SHA = None  # Final result notebook pins the verified implementation commit.
PLAN_SHA = '50e7d70cf1eb1fe892c144a0721dbf490e07b656'
REFERENCE_SHA = '0f8d134a41fe84fb1e79cdf43a91b7c2f77f67dc'
BASELINE = Path('/content/drive/MyDrive/time-entry-portfolio-lab/daily_stop/baseline_cc32f32e3df5/daily_stop_baseline_trades.csv')
M1_ROOT = Path('/content/drive/MyDrive')
OUT = Path('/content')
MOUNT_INPUT_DRIVE = True
if MOUNT_INPUT_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')


In [ ]:
import urllib.request, sys, subprocess
assert RESEARCH_SHA and len(RESEARCH_SHA)==40, 'Set the verified implementation SHA from the run record.'
stage=Path('/content/trend_strength_phase1_code');stage.mkdir(exist_ok=True)
base='https://raw.githubusercontent.com/TR-KJ/time-entry-portfolio-lab/'
for name in ['src/research/trend_strength_phase1.py','src/research/trend_strength_phase1_frozen_inputs.json','tests/test_trend_strength_phase1.py','tests/verify_trend_strength_phase1.py']:
    (stage/Path(name).name).write_bytes(urllib.request.urlopen(base+RESEARCH_SHA+'/'+name).read())
for name in ['src/research/volatility_phase1.py','src/research/volatility_phase1_frozen_inputs.json']:
    (stage/Path(name).name).write_bytes(urllib.request.urlopen(base+REFERENCE_SHA+'/'+name).read())
sys.path.insert(0,str(stage))
subprocess.run([sys.executable,'-m','unittest','discover','-s',str(stage),'-p','test_trend_strength_phase1.py','-v'],check=True)
from trend_strength_phase1 import run
result=run(BASELINE,M1_ROOT,OUT,RESEARCH_SHA)


In [ ]:
from IPython.display import display
print('Portfolio: LOW / NORMAL / HIGH, pooled / strategy equal-weighted')
g=result['group_summary'];display(g[g.Group=='Portfolio'])
display(result['combined_decision']);display(result['decision'])
print('全28戦略: Primary ADX14 / Robustness ER20')
for name in ['strategy_primary','strategy_robustness','group_summary','period_summary','regime_coverage','manual_audit','verification','run_record']:
    print(name);display(result[name])


## 解釈
2022–2026は既閲覧でfresh holdoutではありません。CIは暦週cluster5000回seed20260913、週跨ぎ依存・非定常性は完全には扱いません。多重比較未調整。
TrendとVolatilityは相関し得るため独立情報・因果・将来filter有効性の証明ではありません。
Portfolio BOTH_SUPPORTEDの場合のみ、別事前登録のPhase2で五分位とVolatilityを揃えた追加情報を検証します。
後段Economic Valueは R2 vs R2 + Trend Strength。今回Volatilityとの交差分析・統合はしません。
詳細assignment/dailyは/contentのみ。GitHubへの掲載は軽量audit。


In [ ]:
SAVE_TO_DRIVE = False
if SAVE_TO_DRIVE:
    import shutil
    target=Path('/content/drive/MyDrive/time-entry-portfolio-lab/trend_strength_phase1');target.mkdir(parents=True,exist_ok=True)
    for p in OUT.glob('trend_strength_phase1_*.csv'):
        shutil.copy2(p,target/p.name)
